# EDA를 통한 각 지표 확인
## 데이터 구조 점검 및 변수 이해
### 데이터 준비

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/마늘_이상치제거_주간기준_등급코드.csv', encoding='cp949')
df_grow = pd.read_csv('data/factor_external_weekly.csv', encoding='utf-8')

In [ ]:
# item_code 생성
df['item_code'] = '1209'  # 거래데이터에는 대분류 코드가 없고 한 종류만 있음

df_grow['item_code'] = df_grow['item_code'].astype(str) 
df_grow['item_code'] = df_grow['item_code'].str.zfill(4)

In [ ]:
# weekno 열을 만들고 주차 표기를 통일시켜 merge 준비
def get_week_of_year(date):
    date = pd.to_datetime(date)
    year = date.year
    week_number = date.isocalendar().week
    return f"{year}{week_number:02d}"

df['weekno'] = df['연월일'].apply(get_week_of_year)

In [ ]:
def format_week_int(week_no):
    # week_no가 202401, 202402 등 정수형이면
    week_no = int(week_no)
    year = week_no // 100
    week = week_no % 100
    return f"{year}{week:02d}"

df_grow['weekno'] = df_grow['week_no'].apply(format_week_int)

In [ ]:
# 기 생성된 휴일여부	명절지수	작기정보 칼럼 삭제
df.drop(columns=['휴일여부', '명절지수', '작기정보'], inplace=True)

In [ ]:
# 병합하기
merged_df  = pd.merge( df,
                    df_grow[['weekno', 'item_code', 'holiday_flag', 'holiday_score', 'grow_score']],
                    left_on=['weekno', 'item_code'],
                    right_on=['weekno', 'item_code'],
                    how='left'
                )
merged_df.head()

In [ ]:
# 강수량, 1시간최고 강수량은 결측치(-9) 혹은 비가 안옴(0)이 많아 0 이하는 0으로 처리
# merged_df['강수량(mm)'] = merged_df['강수량(mm)'<=0].count()
merged_df.loc[merged_df['강수량(mm)']<=0, '강수량(mm)'] = 0
merged_df.loc[merged_df['1시간최고강수량(mm)']<=0, '1시간최고강수량(mm)'] = 0

In [ ]:
# 일간 거래 데이터 (필요한 열만 사용)
df_daily_galic = merged_df.loc[:, ['연월일', '품종코드', '등급코드', '총거래량(kg)','주간평균단가(원)','직팜산지코드','일평균기온','최고기온','최저기온','평균상대습도','강수량(mm)','1시간최고강수량(mm)', 'holiday_flag', 'holiday_score','grow_score', '총금액(원)']]
df_daily_galic.head()

In [ ]:
# 주간 거래 데이터 병합
df_galic = merged_df.loc[:, ['weekno', '품종코드', '등급코드', '총금액(원)', '총거래량(kg)','직팜산지코드','일평균기온','최고기온','최저기온','평균상대습도','강수량(mm)','1시간최고강수량(mm)', 'holiday_flag', 'holiday_score','grow_score']]
df_weekly_galic = df_galic.groupby(['weekno', '품종코드', '등급코드', '직팜산지코드']).agg({
    '총금액(원)' : 'sum', 
    '총거래량(kg)' : 'sum', 
    '일평균기온': 'mean', 
    '최고기온': 'max', 
    '최저기온' : 'min', 
    '평균상대습도' : 'mean', 
    '강수량(mm)' : 'sum', 
    '1시간최고강수량(mm)' : 'sum', 
    'holiday_flag' : 'sum', 
    'holiday_score' : 'sum', 
    'grow_score' : 'sum'
}).reset_index()

In [ ]:
# 주간 단가 계산
df_weekly_galic['평균단가(원)'] = round(df_weekly_galic['총금액(원)'] / df_weekly_galic['총거래량(kg)'])

# weekno에서 year, week 분리 (문자열 슬라이싱)
df_weekly_galic['year'] = df_weekly_galic['weekno'].astype(str).str[:4]
df_weekly_galic['week'] = df_weekly_galic['weekno'].astype(str).str[4:]

# 주간 시작일 입력 (시계열 특성 - prophet)
import datetime

def year_week_to_date(year, week):
    # ISO 주차는 매년 첫 번째 주의 월요일이 기준
    return datetime.date.fromisocalendar(int(year), int(week), 1)  # 1: 월요일

df_weekly_galic['week_start'] = df_weekly_galic.apply(lambda row: year_week_to_date(row['year'], row['week']), axis=1)
df_weekly_galic['week_start'] = pd.to_datetime(df_weekly_galic['week_start'])
df_weekly_galic.drop(columns='weekno', inplace=True)

In [ ]:
# 저장
# df_daily_galic.to_csv('data/trade_daily_galic.csv', encoding='cp949', index=False)
df_weekly_galic.to_csv('data/trade_weekly_galic.csv', encoding='cp949', index=False)

# 샘플파일 생성 (chat과 편안한 상담용 )
# df_daily_galic_sample = df_daily_galic.iloc[:100]
# df_daily_galic_sample.to_csv('data/trade_daily_galic_sample.csv', encoding='cp949', index=False)
# df_weekly_galic_sample = df_weekly_galic.iloc[:100]
# df_weekly_galic_sample.to_csv('data/trade_weekly_galic_sample.csv', encoding='cp949', index=False)

In [ ]:
df_weekly_galic

In [ ]:
df = pd.concat([df_weekly_galic.drop(columns=['평균단가(원)', '총금액(원)']), df_weekly_galic.iloc[:, -4]], axis=1)
# df = pd.concat([df_weekly_galic_sample.drop(columns=['평균단가(원)', '총금액(원)']), df_weekly_galic_sample.iloc[:, -4]], axis=1)

In [ ]:
df.to_csv('data/trade_weekly_galic.csv', encoding='cp949', index=False)

# 모델링

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_weekly_galic = pd.read_csv('data/trade_weekly_galic.csv', encoding='cp949')
# df_weekly_galic_sample = pd.read_csv('data/trade_weekly_galic_sample.csv', encoding='cp949')

In [ ]:
print(df_weekly_galic.isna().sum())  # 대체로 산지가 없거나 수입산인 경우 기후 데이터 없음
df_weekly_galic_drop = df_weekly_galic.dropna()

In [ ]:
df_weekly_galic.info()

In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
# 데이터 로드
df = pd.read_csv('data/trade_weekly_galic.csv', encoding='cp949', parse_dates=['week_start'])
print(df.shape)
df.head()

In [ ]:
# 결측치 처리 (간단히 결측치 행 제거)
df = df.dropna()
print(df.isna().sum())

In [ ]:
# feature/target 설정
X = df.drop(['평균단가(원)', 'week_start'], axis=1)
y = df['평균단가(원)']

In [ ]:
# 범주형 변수 인코딩 (one-hot)
X = pd.get_dummies(X, columns=['품종코드', '등급코드', '직팜산지코드', 'year', 'week'])

In [ ]:
# 학습/테스트 분리 (시계열 데이터라면 shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(X_train.shape, X_test.shape)

In [ ]:
# 모델 정의
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
lgbm = LGBMRegressor(n_estimators=200, random_state=42, n_jobs=-1)

In [ ]:
# 학습
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
lgbm.fit(X_train, y_train)

In [ ]:
# 예측
pred_rf = rf.predict(X_test)
pred_xgb = xgb.predict(X_test)
pred_lgbm = lgbm.predict(X_test)

# 앙상블 (단순평균)
pred_ensemble = (pred_rf + pred_xgb + pred_lgbm) / 3

In [ ]:
# 평가
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
rmse_lgbm = np.sqrt(mean_squared_error(y_test, pred_lgbm))
rmse_ensemble = np.sqrt(mean_squared_error(y_test, pred_ensemble))

print(f'RF RMSE: {rmse_rf:.2f}')
print(f'XGB RMSE: {rmse_xgb:.2f}')
print(f'LGBM RMSE: {rmse_lgbm:.2f}')
print(f'Ensemble RMSE: {rmse_ensemble:.2f}')

# 모델 고도화

In [ ]:
# 1. 라이브러리 임포트
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import shap

In [ ]:
# 2. 데이터 로드 및 전처리
df = pd.read_csv('data/trade_weekly_galic.csv', encoding='cp949', parse_dates=['week_start'])
df = df.dropna()
X = df.drop(['평균단가(원)', 'week_start'], axis=1)
y = df['평균단가(원)']
X = pd.get_dummies(X, columns=['품종코드', '등급코드', '직팜산지코드', 'year', 'week'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
# 3. 하이퍼파라미터 튜닝 (예시: RandomForest)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_rf = GridSearchCV(rf, param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Best RF params:", grid_rf.best_params_)
best_rf = grid_rf.best_estimator_

In [ ]:
# 4. XGBoost, LightGBM 간단 튜닝 (필요시 GridSearchCV로 확장)
xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
lgbm = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
lgbm.fit(X_train, y_train)

In [ ]:
# 5. Stacking 앙상블
stack = StackingRegressor(
    estimators=[
        ('rf', best_rf),
        ('xgb', xgb),
        ('lgbm', lgbm)
    ],
    final_estimator=RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
)
stack.fit(X_train, y_train)

In [ ]:
# 6. 예측 및 평가
pred_rf = best_rf.predict(X_test)
pred_xgb = xgb.predict(X_test)
pred_lgbm = lgbm.predict(X_test)
# pred_stack = stack.predict(X_test)  # 스태킹 사용 시 활성화
pred_ensemble = (pred_rf + pred_xgb + pred_lgbm) / 3

print(f'RF RMSE: {np.sqrt(mean_squared_error(y_test, pred_rf)):.2f}')
print(f'XGB RMSE: {np.sqrt(mean_squared_error(y_test, pred_xgb)):.2f}')
print(f'LGBM RMSE: {np.sqrt(mean_squared_error(y_test, pred_lgbm)):.2f}')
print(f'Ensemble RMSE: {np.sqrt(mean_squared_error(y_test, pred_ensemble)):.2f}')
# print(f'Stacking RMSE: {np.sqrt(mean_squared_error(y_test, pred_stack)):.2f}')  # 스태킹 사용 시 활성화

In [ ]:
# 7. 변수 중요도 시각화 (예: RandomForest)
importances = best_rf.feature_importances_
indices = np.argsort(importances)[-20:]  # 상위 20개
plt.figure(figsize=(8, 8))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.title("RandomForest Feature Importances (Top 20)")
plt.show()

In [ ]:
# 8. SHAP 해석 (예: XGBoost)
explainer = shap.Explainer(xgb, X_train)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test, max_display=20)